# Figure 6 — passivated and unpassivated depth profiles

This notebook reproduces the corresponding numbered manuscript figure using the numerical inputs stated below.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src'/'surpass').exists()), None)
if root is None:
    raise FileNotFoundError('Could not locate the SURPASS project root.')
sys.path.insert(0, str(root/'src'))

import surpass
from surpass import (
    BulkModel, BulkTransportTable, InterfaceDefectModel, Layer, OpticalStack,
    SteadyState1DSolver, SurfaceBoundaryTable, SurfaceSRHModel, Wafer,
    get_optical_material, reconstruct_band_profile,
)

print('surpass version:', surpass.__version__)
print('loaded from:', Path(surpass.__file__).resolve())
if tuple(map(int, surpass.__version__.split('.')[:3])) < (0,9,0):
    raise ImportError('This notebook requires surpass 1.0.0 or newer.')

try:
    import scienceplots
    plt.style.use(['science','notebook','no-latex'])
except ImportError:
    plt.style.use('default')

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 10.5,
    'axes.linewidth': 1.15,
    'axes.spines.top': True,
    'axes.spines.right': True,
    'xtick.major.width': 1.05,
    'ytick.major.width': 1.05,
    'xtick.minor.width': 0.85,
    'ytick.minor.width': 0.85,
})

output_dir = Path.cwd()/'output'
output_dir.mkdir(parents=True,exist_ok=True)


## 1. Model assumptions

The passivated case uses $D_{it}=3\times10^{10}$ eV$^{-1}$cm$^{-2}$ and
$Q_f/q=+2\times10^{12}$ cm$^{-2}$. The unpassivated case uses
$D_{it}=10^{12}$ eV$^{-1}$cm$^{-2}$ and $Q_f=0$. These are illustrative model
coordinates rather than fitted or independently extracted values.

Both cases use the same optical generation profile so that the comparison
isolates the electrical interface condition. The nominal calculation retains
the 5-ns bulk SRH lifetime and photon-recycling probability of 0.93 used in the
paper model. The electron and hole capture cross sections are the
literature-motivated InP starting values from Adamowicz *et al.*, not measured
PO$_x$/InP parameters.


In [ ]:
WAVELENGTH_NM = 514.0
THICKNESS_UM = 620.0
THICKNESS_CM = THICKNESS_UM*1e-4

CASES = {
    'passivated': dict(dit=3e10, qf_cm2=2e12, ls='-', lw=2.25),
    'unpassivated': dict(dit=1e12, qf_cm2=0.0,
                         ls=(0,(1.2,2.0)), lw=1.55),
}

SIGMA_N_CM2 = 1e-14
SIGMA_P_CM2 = 1e-13
VTH_N_CM_S = 4.13e7
VTH_P_CM_S = 1.51e7
REAR_S_CM_S = 1e5
ENABLE_BGN = False
BULK_SRH_LIFETIME_S = 5e-9
PHOTON_RECYCLING_PROBABILITY = 0.93

ELECTRON_COLOR = '#0072B2'  # colorblind-safe blue
HOLE_COLOR = '#D55E00'      # colorblind-safe vermillion
EC_COLOR = '#222222'
EV_COLOR = '#888888'
GENERATION_COLOR = '#009E73'
RAD_COLOR = '#009E73'
AUGER_COLOR = '#CC79A7'
SRH_COLOR = '#8C564B'

wafer_settings = {
    'n-InP':dict(wafer=Wafer('InP',donor_cm3=8.5e18,thickness_um=THICKNESS_UM),
                 intensity_w_cm2=1.0),
    'p-InP':dict(wafer=Wafer('InP',acceptor_cm3=5.1e18,thickness_um=THICKNESS_UM),
                 intensity_w_cm2=13.0),
}

air=get_optical_material('air')
inp_optical=get_optical_material('InP_demo')
pox_optical=get_optical_material('POx_demo')
alox_optical=get_optical_material('AlOx_demo')
stack=OpticalStack(
    air,
    [Layer(alox_optical,40.0,'AlOx'), Layer(pox_optical,4.0,'POx')],
    inp_optical,
)
optical=stack.solve(WAVELENGTH_NM)

display(pd.DataFrame([
    {'case':name, 'Dit (eV^-1 cm^-2)':case['dit'],
     'Qf/q (cm^-2)':case['qf_cm2']}
    for name,case in CASES.items()
]))
display(pd.DataFrame([{
    'wavelength (nm)':WAVELENGTH_NM,
    'reflectance':optical.reflectance,
    'substrate entry fraction':optical.substrate_entry_fraction,
    'absorption depth (nm)':optical.absorption_depth_nm,
}]))


## 2. Coupled solutions and smooth deep-tail display

Four independent steady-state solutions are calculated: passivated and
unpassivated surfaces for each wafer polarity. The transport solution itself
is not smoothed.

Two display-side reconstructions are used. First, the quasi-neutral transport
injection is interpolated through the thin space-charge region. The
Poisson-derived electron and hole redistribution factors are then applied to
that local injection. This removes the artificial discontinuity produced when
the entire space-charge profile is reconstructed using only the surface
injection and then joined to a depth-dependent transport solution.

Second, only the numerically unresolved far tail used for plotting is continued with
the analytical low-injection diffusion asymptote. The previous implementation
switched to that asymptote abruptly at one mesh point, which could produce a
small visible kink in $p(z)$ for n-InP. Here the numerical and asymptotic
profiles are blended in log space with a smoothstep function over the final
ten finely resolved cells. This matches both the value and slope continuously
at the transition and does not alter the recombination-active region.


In [ ]:
edges_cm=np.r_[0.0,np.geomspace(1e-8,5e-4,120),
               np.linspace(5e-4,THICKNESS_CM,121)[1:]]
solutions={}
summary=[]

for label,settings in wafer_settings.items():
    bulk=BulkModel(settings['wafer'],enable_bgn=ENABLE_BGN,
                   tau_srh_s=BULK_SRH_LIFETIME_S)
    transport=BulkTransportTable.from_bulk_model(bulk,1.0,1e21,points=181)
    solver=SteadyState1DSolver(transport)
    generation=stack.generation_cell_average_cm3_s(
        WAVELENGTH_NM,settings['intensity_w_cm2'],edges_cm)
    solutions[label]={}

    for case_name,case in CASES.items():
        defects=InterfaceDefectModel(
            case['dit'],SIGMA_N_CM2,SIGMA_P_CM2,VTH_N_CM_S,VTH_P_CM_S,
            shape='constant',energy_points=201)
        surface=SurfaceSRHModel(
            bulk,case['qf_cm2'],defects,electrostatics='self_consistent',
            driving_force='excess_product')
        boundary=SurfaceBoundaryTable.from_surface_model(
            surface,1.0,1e21,points=101)
        result=solver.solve(
            THICKNESS_CM,generation,boundary,REAR_S_CM_S,
            cell_edges_cm=edges_cm,max_nfev=2500,
            recycling_probability=PHOTON_RECYCLING_PROBABILITY)
        if not result.converged:
            raise RuntimeError(f'{label}, {case_name}: solution did not converge')
        surface_state=surface.state(result.front_surface_delta_cm3)
        profile=reconstruct_band_profile(
            surface,result.front_surface_delta_cm3,points=1201,
            tail_fraction=1e-7)
        solutions[label][case_name]=dict(
            bulk=bulk,transport=transport,surface=surface,result=result,
            surface_state=surface_state,profile=profile,
        )
        summary.append({
            'wafer':label,
            'case':case_name,
            'average delta n (cm^-3)':result.average_delta_n_cm3,
            'surface delta n (cm^-3)':result.front_surface_delta_cm3,
            'psi_s (V)':surface_state.psi_surface_v,
            'n_s (cm^-3)':surface_state.n_surface_cm3,
            'p_s (cm^-3)':surface_state.p_surface_cm3,
            'front Seff (cm/s)':result.front_effective_s_cm_s,
            'balance error':result.relative_balance_error,
        })

summary=pd.DataFrame(summary)
display(summary)
assert max(abs(summary['balance error'])) < 1e-5


def qfl_profiles(bulk,result,delta_profile):
    states=[bulk.injection(max(float(d),1e-300)) for d in delta_profile]
    eg=bulk.equilibrium().eg_effective_ev
    vt=bulk.wafer.temperature_k*8.617333262145e-5
    efn=np.array([eg+vt*s.eta_n for s in states])
    efp=np.array([-vt*s.eta_p for s in states])
    return efn,efp


def composite_scr_profiles(item):
    """Combine SCR electrostatics with the local transport injection smoothly."""
    bulk,result,profile=item['bulk'],item['result'],item['profile']
    equilibrium=bulk.equilibrium()
    x_transport=result.depth_cm
    x_scr=profile.depth_cm

    # Include the exact boundary injection at z=0 before interpolating the
    # finite-volume, cell-centred transport result through the SCR.
    x_nodes=np.r_[0.0,x_transport]
    delta_nodes=np.r_[result.front_surface_delta_cm3,item['resolved_delta']]
    delta_scr=np.interp(x_scr,x_nodes,delta_nodes)

    # The Poisson profile supplies electrostatic redistribution factors. Their
    # denominators are the neutral concentrations at the surface injection.
    neutral_surface_n=equilibrium.n0_cm3+result.front_surface_delta_cm3
    neutral_surface_p=equilibrium.p0_cm3+result.front_surface_delta_cm3
    electron_factor=profile.n_cm3/neutral_surface_n
    hole_factor=profile.p_cm3/neutral_surface_p
    n_scr=electron_factor*(equilibrium.n0_cm3+delta_scr)
    p_scr=hole_factor*(equilibrium.p0_cm3+delta_scr)
    efn_scr,efp_scr=qfl_profiles(bulk,result,delta_scr)
    return delta_scr,n_scr,p_scr,efn_scr,efp_scr


def smooth_resolved_tail(item,transition_cells=10):
    result,table=item['result'],item['transport']
    original=np.maximum(result.delta_n_cm3.copy(),1e-300)
    delta=original.copy()
    x=result.depth_cm
    fine=np.flatnonzero(result.cell_width_cm < 1e-4)  # cell width < 1 um
    end=int(fine[-1])
    start=int(fine[max(0,len(fine)-transition_cells)])

    low=table.evaluate(np.array([1.0]))
    inverse_tau=float((1-PHOTON_RECYCLING_PROBABILITY)*low['r_rad'][0]
                      +low['r_auger'][0]+low['r_srh'][0])
    low_length=np.sqrt(float(low['d_amb'][0])/inverse_tau)

    analytic=original[start]*np.exp(
        np.clip(-(x[start:]-x[start])/low_length,-745.0,0.0))
    if end > start:
        t=(x[start:end+1]-x[start])/(x[end]-x[start])
        weight=t*t*(3-2*t)  # zero derivative at both blend boundaries
        log_blend=((1-weight)*np.log(original[start:end+1])
                   +weight*np.log(np.maximum(analytic[:end-start+1],1e-300)))
        delta[start:end+1]=np.exp(log_blend)
    delta[end+1:]=analytic[end+1-start:]
    return delta,low_length,(start,end)


for wafer_cases in solutions.values():
    for item in wafer_cases.values():
        (item['resolved_delta'],item['low_injection_length_cm'],
         item['tail_blend_indices'])=smooth_resolved_tail(item)
        (item['delta_scr'],item['n_scr'],item['p_scr'],
         item['efn_scr'],item['efp_scr'])=composite_scr_profiles(item)


## 3. Publication figure

Solid lines show the passivated case; thin dotted lines show the unpassivated
case. The optical generation profile is common to both interface cases for a
given wafer. Recombination tables report sheet-integrated loss rates in
cm$^{-2}$s$^{-1}$ and their percentage of the total recombination. The small
rear-surface term is retained so that each percentage column sums to 100%.


In [ ]:
fig,axes=plt.subplots(3,2,figsize=(14.6,11.2),sharex='col',sharey='row',
                      constrained_layout=True)
generation_twins=[]

# Common ordinate limits are calculated from both wafers and both cases.
energy_values=[]
carrier_values=[]
rate_values=[]
generation_values=[]
for label,wafer_cases in solutions.items():
    for case_name,item in wafer_cases.items():
        profile=item['profile']; result=item['result']; bulk=item['bulk']
        equilibrium=bulk.equilibrium(); delta=item['resolved_delta']
        efn,efp=qfl_profiles(bulk,result,delta)
        energy_values.extend([profile.ec_ev,profile.ev_ev,efn,efp])
        carrier_values.extend([profile.n_cm3,profile.p_cm3,
                               equilibrium.n0_cm3+delta,
                               equilibrium.p0_cm3+delta])
        rate_values.extend([result.r_rad_cm3_s,result.r_auger_cm3_s,
                            result.r_srh_cm3_s])
        generation_values.append(result.generation_cm3_s)

energy_min=min(np.nanmin(v) for v in energy_values)
energy_max=max(np.nanmax(v) for v in energy_values)
energy_pad=.05*(energy_max-energy_min)
carrier_min=max(1.0,min(np.nanmin(v[v>0]) for v in carrier_values))
carrier_max=max(np.nanmax(v) for v in carrier_values)
rate_max=max(np.nanmax(v) for v in rate_values)
rate_floor=max(1.0,rate_max*1e-11)
generation_max=max(np.nanmax(v) for v in generation_values)
generation_floor=max(1.0,generation_max*1e-12)

for col,(label,settings) in enumerate(wafer_settings.items()):
    for case_name,case in CASES.items():
        item=solutions[label][case_name]
        bulk,result,profile=item['bulk'],item['result'],item['profile']
        equilibrium=bulk.equilibrium()
        x_um=result.depth_cm*1e4
        x_scr_um=profile.depth_cm*1e4
        delta=item['resolved_delta']
        efn_bulk,efp_bulk=qfl_profiles(bulk,result,delta)
        bulk_after_scr=x_um>x_scr_um[-1]

        # Bands and quasi-Fermi levels.
        ax=axes[0,col]
        x_full=np.r_[x_scr_um,x_um[bulk_after_scr]]
        ec_full=np.r_[profile.ec_ev,
                      np.full(bulk_after_scr.sum(),equilibrium.eg_effective_ev)]
        ev_full=np.r_[profile.ev_ev,np.zeros(bulk_after_scr.sum())]
        efn_full=np.r_[item['efn_scr'],efn_bulk[bulk_after_scr]]
        efp_full=np.r_[item['efp_scr'],efp_bulk[bulk_after_scr]]
        ax.plot(x_full,ec_full,color=EC_COLOR,ls=case['ls'],lw=case['lw'],
                alpha=.95)
        ax.plot(x_full,ev_full,color=EV_COLOR,ls=case['ls'],lw=case['lw'],
                alpha=.95)
        ax.plot(x_full,efn_full,color=ELECTRON_COLOR,ls=case['ls'],lw=case['lw'])
        ax.plot(x_full,efp_full,color=HOLE_COLOR,ls=case['ls'],lw=case['lw'])

        # Surface-region populations followed by quasi-neutral transport.
        ax=axes[1,col]
        n_transport=equilibrium.n0_cm3+delta
        p_transport=equilibrium.p0_cm3+delta
        n_full=np.r_[item['n_scr'],n_transport[bulk_after_scr]]
        p_full=np.r_[item['p_scr'],p_transport[bulk_after_scr]]
        ax.semilogy(x_full,n_full,color=ELECTRON_COLOR,ls=case['ls'],lw=case['lw'])
        ax.semilogy(x_full,p_full,color=HOLE_COLOR,ls=case['ls'],lw=case['lw'])

        # Volumetric bulk-recombination profiles.
        ax=axes[2,col]
        channels=[(result.r_rad_cm3_s,RAD_COLOR,r'$R_{rad}$'),
                  (result.r_auger_cm3_s,AUGER_COLOR,r'$R_{Auger}$'),
                  (result.r_srh_cm3_s,SRH_COLOR,r'$R_{SRH,bulk}$')]
        for values,color,_ in channels:
            visible=values>=rate_floor
            if np.any(visible):
                ax.loglog(x_um[visible],values[visible],color=color,
                          ls=case['ls'],lw=case['lw'])

    # Generation is independent of electrical interface condition.
    pass_result=solutions[label]['passivated']['result']
    x_um=pass_result.depth_cm*1e4
    axg=axes[1,col].twinx()
    visible=pass_result.generation_cm3_s>=generation_floor
    axg.semilogy(x_um[visible],pass_result.generation_cm3_s[visible],
                 color=GENERATION_COLOR,lw=1.8,alpha=.85)
    axg.set_ylim(generation_floor,generation_max*1.8)
    axg.set_ylabel(r'$G(z)$ (cm$^{-3}$ s$^{-1}$)',color=GENERATION_COLOR)
    axg.tick_params(axis='y',colors=GENERATION_COLOR,width=1.05)
    generation_twins.append(axg)

    # Opaque table inset avoids line/text collisions and compares both cases.
    table_rows=[]
    row_labels=['Surface','Radiative','Bulk SRH','Auger','Rear']
    attributes=[
        'front_surface_recombination_cm2_s',
        'bulk_radiative_recombination_cm2_s',
        'bulk_srh_recombination_cm2_s',
        'bulk_auger_recombination_cm2_s',
        'rear_surface_recombination_cm2_s',
    ]
    for attr in attributes:
        row=[]
        for case_name in ('passivated','unpassivated'):
            result=solutions[label][case_name]['result']
            value=getattr(result,attr)
            percentage=100*value/result.total_recombination_cm2_s
            if value == 0:
                flux_text='$0$'
            else:
                exponent=int(np.floor(np.log10(abs(value))))
                mantissa=value/10**exponent
                flux_text=rf'${mantissa:.2f}\times10^{{{exponent}}}$'
            row.extend([flux_text,f'{percentage:.1f}'])
        table_rows.append(row)

    ax=axes[2,col]
    inset=ax.inset_axes([.16,.025,.55,.44])
    inset.set_facecolor('white')
    inset.patch.set_alpha(.94)
    inset.set_xticks([]); inset.set_yticks([])
    for spine in inset.spines.values():
        spine.set_color('.55'); spine.set_linewidth(.7)
    table=inset.table(
        cellText=table_rows,rowLabels=row_labels,
        colLabels=['Pass. flux','Pass. %','Unpass. flux','Unpass. %'],
        cellLoc='right',rowLoc='left',bbox=[0,0,1,1])
    table.auto_set_font_size(False)
    table.set_fontsize(6.6)
    table.scale(1.0,1.12)
    for key,table_cell in table.get_celld().items():
        table_cell.set_edgecolor('.78')
        table_cell.set_linewidth(.45)
        if key[0]==0:
            table_cell.set_facecolor('.92')
            table_cell.get_text().set_weight('bold')
        if key[1]==-1:
            table_cell.get_text().set_weight('bold')

# Shared formatting and explicit common y limits.
for col,label in enumerate(wafer_settings):
    axes[0,col].set_title(label,fontsize=14,fontweight='bold')
    axes[0,col].set_ylim(energy_min-energy_pad,energy_max+energy_pad)
    axes[1,col].set_ylim(carrier_min/1.5,carrier_max*1.5)
    axes[2,col].set_ylim(rate_floor,rate_max*2)
    axes[2,col].set_xlabel(r'Depth into InP ($\mu$m)')

axes[0,0].set_ylabel(r'Energy relative to bulk $E_V$ (eV)')
axes[1,0].set_ylabel(r'Carrier concentration (cm$^{-3}$)')
axes[2,0].set_ylabel(r'Bulk excess recombination (cm$^{-3}$ s$^{-1}$)')

for ax in axes.ravel():
    ax.set_xscale('symlog',linthresh=1e-3,linscale=1.0)
    ax.set_xlim(0,THICKNESS_UM)
    ax.tick_params(which='both',direction='in',top=True,right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.15)

# The middle panels use a green twinned generation axis on the right. Remove
# the underlying black right ticks/spine from the carrier axes.
for col in range(2):
    axes[1,col].tick_params(axis='y',right=False,labelright=False)
    axes[1,col].spines['right'].set_visible(False)
    generation_twins[col].spines['right'].set_color(GENERATION_COLOR)

# Compact legends explain quantity colors and case line styles separately.
energy_handles=[
    Line2D([],[],color=EC_COLOR,lw=2.2,label=r'$E_C$'),
    Line2D([],[],color=EV_COLOR,lw=2.2,label=r'$E_V$'),
    Line2D([],[],color=ELECTRON_COLOR,lw=2.2,label=r'$E_{Fn}$'),
    Line2D([],[],color=HOLE_COLOR,lw=2.2,label=r'$E_{Fp}$'),
]
quantity_legend=axes[0,0].legend(
    handles=energy_handles,ncol=2,fontsize=7.7,loc='upper left',
    frameon=True,framealpha=.92,facecolor='white',edgecolor='.65')
axes[0,0].add_artist(quantity_legend)

condition_handles=[
    Line2D([],[],color='black',lw=2.2,ls='-',
           label=r'Passivated: $D_{it}=3\times10^{10}$, $Q_f/q=+2\times10^{12}$'),
    Line2D([],[],color='black',lw=1.55,ls=(0,(1.2,2.0)),
           label=r'Unpassivated: $D_{it}=10^{12}$, $Q_f=0$'),
]
axes[0,0].legend(handles=condition_handles,ncol=1,fontsize=7.3,loc='lower left',
                 frameon=True,framealpha=.92,facecolor='white',edgecolor='.65',
                 title=r'Interface assumptions (eV$^{-1}$ cm$^{-2}$; cm$^{-2}$)',
                 title_fontsize=7.1)

carrier_handles=[
    Line2D([],[],color=ELECTRON_COLOR,lw=2.2,label=r'$n(z)$'),
    Line2D([],[],color=HOLE_COLOR,lw=2.2,label=r'$p(z)$'),
    Line2D([],[],color=GENERATION_COLOR,lw=1.8,label=r'$G(z)$'),
]
axes[1,0].legend(handles=carrier_handles,ncol=3,fontsize=7.8,loc='best',
                 frameon=True,framealpha=.92,facecolor='white',edgecolor='.65')

rate_handles=[
    Line2D([],[],color=RAD_COLOR,lw=2.2,label=r'$R_{rad}$'),
    Line2D([],[],color=AUGER_COLOR,lw=2.2,label=r'$R_{Auger}$'),
    Line2D([],[],color=SRH_COLOR,lw=2.2,label=r'$R_{SRH,bulk}$'),
]
for col in range(2):
    axes[2,col].legend(handles=rate_handles,ncol=1,fontsize=7.4,loc='upper right',
                 frameon=True,framealpha=.92,facecolor='white',edgecolor='.65')

fig.savefig(output_dir/'paper_inp_passivated_unpassivated_profiles.png',
            bbox_inches='tight')
fig.savefig(output_dir/'paper_inp_passivated_unpassivated_profiles.pdf',
            bbox_inches='tight')
plt.show()
